## Mamba-2 — Inference Notebook

Load the trained Mamba-2 checkpoint and generate text from prompts.

## 1 · Imports & Config

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn.functional as F
from tokenizers import Tokenizer
from model import Mamba2Model

# ── Model hyper-parameters (must match training) ────────────────────
VOCAB_SIZE = 4096
D_MODEL    = 256
N_LAYERS   = 6
D_STATE    = 16
D_CONV     = 4
EXPAND     = 2
HEADDIM    = 64
CHUNK_SIZE = 64
NGROUPS    = 1

# ── Special tokens ─────────────────────────────────────────────────
BOS_ID = 2
EOS_ID = 3

# ── Paths ──────────────────────────────────────────────────────────
CHECKPOINT_PATH = "../checkpoints/mamba2_best.pt"
TOKENIZER_PATH  = "../tokenizer_4k.json"

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 2 · Load Tokenizer

In [ ]:
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
print(f"Tokenizer loaded  ·  vocab size = {tokenizer.get_vocab_size()}")

## 3 · Load Model & Checkpoint

In [ ]:
model = Mamba2Model(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layer=N_LAYERS,
    expand=EXPAND,
    headdim=HEADDIM,
    d_state=D_STATE,
    chunk_size=CHUNK_SIZE,
    d_conv=D_CONV,
    ngroups=NGROUPS
).to(DEVICE)

state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded  ·  {n_params / 1e6:.2f}M parameters")

## 4 · Generation Utilities

In [ ]:
@torch.no_grad()
def generate(
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.95,
):
    """
    Auto-regressive generation with temperature, top-k, and nucleus (top-p) sampling.
    """
    encoded = tokenizer.encode(prompt)
    input_ids = torch.tensor(
        [BOS_ID] + encoded.ids, dtype=torch.long
    ).unsqueeze(0).to(DEVICE)

    for _ in range(max_new_tokens):
        logits = model(input_ids)
        next_logits = logits[:, -1, :] / temperature

        # ── Top-k filtering ────────────────────────────────────────
        if top_k > 0:
            top_k_vals, _ = torch.topk(next_logits, top_k)
            threshold = top_k_vals[:, -1].unsqueeze(-1)
            next_logits = next_logits.masked_fill(next_logits < threshold, -float('inf'))

        # ── Nucleus (top-p) filtering ──────────────────────────────
        if top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            remove_mask = cumulative_probs - F.softmax(sorted_logits, dim=-1) >= top_p
            sorted_logits[remove_mask] = -float('inf')
            next_logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)

        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        input_ids = torch.cat([input_ids, next_token], dim=-1)
        
        # Stop token
        if next_token.item() == EOS_ID:
            break

    return tokenizer.decode(input_ids[0].tolist())


def show(prompt, **kwargs):
    """Pretty-print a generation."""
    text = generate(prompt, **kwargs)
    print(f"\n{'─' * 60}")
    print(f"Prompt │ {prompt}")
    print(f"{'─' * 60}")
    print(text)
    print(f"{'─' * 60}\n")

## 5 · Generate Stories 📖

In [ ]:
show("Once upon a time")

In [ ]:
show("The little cat")

In [ ]:
show("There was a boy named")

## 6 · Experiment with Sampling Parameters

Try different temperatures, top-k, and top-p values to see how generation changes.

In [ ]:
prompt = "Once upon a time"

print("=" * 60)
print("GREEDY  (temperature → 0)")
print("=" * 60)
show(prompt, temperature=0.1, top_k=1)

print("=" * 60)
print("WARM  (temperature = 0.8, top_k = 50)")
print("=" * 60)
show(prompt, temperature=0.8, top_k=50)

print("=" * 60)
print("CREATIVE  (temperature = 1.2, top_p = 0.9)")
print("=" * 60)
show(prompt, temperature=1.2, top_k=0, top_p=0.9)